In [1]:
import pandas as pd
import os 
import matplotlib.pyplot as plt
import glob 
import datetime
import seaborn as sns
import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error 

In [3]:
folder_path = '/Users/alexisdafonseca/Desktop/predict-volcanic-eruptions-ingv-oe/train/'

In [5]:
train = pd.read_csv('train.csv')
print(train.head())

   segment_id  time_to_eruption
0  1136037770          12262005
1  1969647810          32739612
2  1895879680          14965999
3  2068207140          26469720
4   192955606          31072429


In [7]:
mesure = {}
columns = ['sum', 'mean', 'std', 'min', 'max', 'quantile90', 'quantile70', 'quantile50', 'quantile20', 'quantile10']

In [9]:
# Lire un fichier au hasard pour récupérer les colonnes (assumant qu'ils ont tous les mêmes colonnes)
sample_file = f"{folder_path}/{train['segment_id'].iloc[0]}.csv"
sample_data = pd.read_csv(sample_file)

# Initialiser le dictionnaire avec des listes vides pour chaque statistique
for col in sample_data.columns:
    for stat in columns:
        mesure[f"{col}_{stat}"] = []  

# Parcourir chaque segment_id
for file in tqdm.tqdm(train["segment_id"]):
    file_path = f"{folder_path}/{file}.csv"

    try:
        data = pd.read_csv(file_path)

        for column in data.select_dtypes(include=[np.number]).columns:  # Vérifier si la colonne est numérique
            mesure[f"{column}_sum"].append(data[column].sum())
            mesure[f"{column}_mean"].append(data[column].mean())
            mesure[f"{column}_std"].append(data[column].std())
            mesure[f"{column}_max"].append(data[column].max())
            mesure[f"{column}_min"].append(data[column].min())

            # Ajout des quantiles demandés
            mesure[f"{column}_quantile90"].append(data[column].quantile(0.90))
            mesure[f"{column}_quantile70"].append(data[column].quantile(0.70))
            mesure[f"{column}_quantile50"].append(data[column].quantile(0.50))  # Médiane
            mesure[f"{column}_quantile20"].append(data[column].quantile(0.20))
            mesure[f"{column}_quantile10"].append(data[column].quantile(0.10))

        # Gérer les colonnes absentes en remplissant avec NaN
        missing_columns = set(sample_data.columns) - set(data.columns)
        for column in missing_columns:
            for stat in columns:
                mesure[f"{column}_{stat}"].append(np.nan)  # Remplir avec NaN au lieu de 0

    except Exception as e:
        print(f"Erreur lors de la lecture de {file_path}: {e}")
        # Ajouter NaN pour toutes les colonnes en cas d'erreur
        for key in mesure.keys():
            mesure[key].append(np.nan)

# Vérification des longueurs de listes
min_length = min(len(lst) for lst in mesure.values())
for key in mesure.keys():
    mesure[key] = mesure[key][:min_length]  # Troncature des listes trop longues

# Convertir en DataFrame
df_mesure = pd.DataFrame(mesure)
print(df_mesure.head())


100%|███████████████████████████████████████| 4431/4431 [08:07<00:00,  9.09it/s]

   sensor_1_sum  sensor_1_mean  sensor_1_std  sensor_1_min  sensor_1_max  \
0      -96621.0      -1.610323    303.096099       -1421.0        1353.0   
1       85569.0       1.426126    438.360560       -2005.0        2197.0   
2      150278.0       2.504592    241.575415       -1115.0        1148.0   
3      129950.0       2.165797    221.967825       -1025.0        1254.0   
4        4429.0       0.073815    261.695935       -1190.0        1114.0   

   sensor_1_quantile90  sensor_1_quantile70  sensor_1_quantile50  \
0                383.0                153.0                  0.0   
1                528.0                209.0                  0.0   
2                307.0                121.0                  0.0   
3                273.0                108.0                  0.0   
4                329.0                132.0                  0.0   

   sensor_1_quantile20  sensor_1_quantile10  ...  sensor_10_sum  \
0               -249.0               -388.0  ...       155969.0   


In [11]:
train_mesure = pd.DataFrame(mesure)
train_final = pd.concat([train, train_mesure], axis=1)  # Fusionner avec df_main

print(train_final.head())

   segment_id  time_to_eruption  sensor_1_sum  sensor_1_mean  sensor_1_std  \
0  1136037770          12262005      -96621.0      -1.610323    303.096099   
1  1969647810          32739612       85569.0       1.426126    438.360560   
2  1895879680          14965999      150278.0       2.504592    241.575415   
3  2068207140          26469720      129950.0       2.165797    221.967825   
4   192955606          31072429        4429.0       0.073815    261.695935   

   sensor_1_min  sensor_1_max  sensor_1_quantile90  sensor_1_quantile70  \
0       -1421.0        1353.0                383.0                153.0   
1       -2005.0        2197.0                528.0                209.0   
2       -1115.0        1148.0                307.0                121.0   
3       -1025.0        1254.0                273.0                108.0   
4       -1190.0        1114.0                329.0                132.0   

   sensor_1_quantile50  ...  sensor_10_sum  sensor_10_mean  sensor_10_std  \
0  

In [13]:
train_final = train_final.drop(columns = ['segment_id'])
train_final

,time_to_eruption,sensor_1_sum,sensor_1_mean,sensor_1_std,sensor_1_min,sensor_1_max,sensor_1_quantile90,sensor_1_quantile70,sensor_1_quantile50,sensor_1_quantile20,...,sensor_10_sum,sensor_10_mean,sensor_10_std,sensor_10_min,sensor_10_max,sensor_10_quantile90,sensor_10_quantile70,sensor_10_quantile50,sensor_10_quantile20,sensor_10_quantile10
0,12262005,-96621.0,-1.610323,303.096099,-1421.0,1353.0,383.0,153.0,0.0,-249.0,...,155969.0,2.599440,596.865818,-3527.0,2187.0,767.0,324.0,0.0,-503.0,-757.0
1,32739612,85569.0,1.426126,438.360560,-2005.0,2197.0,528.0,209.0,0.0,-333.0,...,1193751.0,19.895518,932.699662,-4591.0,5319.0,1146.0,439.0,0.0,-692.0,-1091.0
2,14965999,150278.0,2.504592,241.575415,-1115.0,1148.0,307.0,121.0,0.0,-195.0,...,58087.0,0.968101,500.544892,-2798.0,2522.0,624.0,249.0,0.0,-402.0,-618.0
3,26469720,129950.0,2.165797,221.967825,-1025.0,1254.0,273.0,108.0,0.0,-171.0,...,502895.0,8.381444,523.449843,-2911.0,3614.0,652.0,254.0,0.0,-400.0,-625.0
4,31072429,4429.0,0.073815,261.695935,-1190.0,1114.0,329.0,132.0,0.0,-216.0,...,-224432.0,-3.740471,562.748316,-2611.0,3274.0,698.0,280.0,0.0,-458.0,-715.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,15695097,54405.0,0.906735,613.427301,-2610.0,2916.0,782.0,322.0,0.0,-511.0,...,-202647.0,-3.377394,1053.539787,-5726.0,5493.0,1305.0,512.0,0.0,-839.0,-1338.0
4427,35659379,476221.0,7.936884,649.089618,-3654.0,4498.0,827.0,329.0,0.0,-521.0,...,656389.0,10.939634,1279.909391,-13017.0,14873.0,1536.0,611.0,0.0,-976.0,-1480.0
4428,31206935,85261.0,1.420993,110.942596,-753.0,557.0,140.0,56.0,0.0,-87.0,...,-93024.0,-1.550374,251.560198,-1708.0,1549.0,291.0,115.0,0.0,-187.0,-293.0
4429,9598270,54350.0,0.905818,478.852581,-3555.0,3938.0,566.0,216.0,0.0,-354.0,...,-355511.0,-6.360564,618.914657,-3588.0,3628.0,757.0,301.0,0.0,-501.0,-776.0


In [15]:
x = pd.DataFrame(train_final.drop(columns = ['time_to_eruption']))
y = pd.DataFrame(train_final['time_to_eruption'])

In [75]:
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size = 0.8)

In [77]:
# Définir l’estimateur XGBoost
xgb_regressor = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

# Définition de l’espace de recherche des hyperparamètres
param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.6, 0.8, 1.0]
}

# Validation croisée rapide (3 folds seulement)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Recherche rapide
random_search = RandomizedSearchCV(
    estimator=xgb_regressor,
    param_distributions=param_grid,
    n_iter=20,  
    scoring='r2',
    cv=kfold,
    verbose=1,
    n_jobs=-1
)

# Exécution de la recherche
random_search.fit(x_train, y_train)

# Meilleurs paramètres trouvés
print("Meilleurs paramètres :", random_search.best_params_)

# Meilleur score R² obtenu
print("Meilleur score R² :", random_search.best_score_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Meilleurs paramètres : {'subsample': 0.6, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.1}
Meilleur score R² : 0.7809065505873078


In [79]:
y_pred = random_search.best_estimator_.predict(x_test)

In [81]:
y_test.shape

(887, 1)

In [83]:
comparison_df = pd.DataFrame({'Valeurs réelles' : y_test.values.flatten(), 'Valeurs prédites' : y_pred.flatten()})
comparison_df

,Valeurs réelles,Valeurs prédites
0,4061438,13026617.0
1,32249219,27320004.0
2,39498290,21600892.0
3,35841678,35427980.0
4,23385989,23041006.0
...,...,...
882,16816678,13688066.0
883,39483628,38995204.0
884,34468280,36639040.0
885,28666827,27101006.0


In [85]:
comparison_df["Valeurs réelles"] = pd.to_timedelta(comparison_df["Valeurs réelles"], unit="s")
comparison_df["Valeurs prédites"] = pd.to_timedelta(comparison_df["Valeurs prédites"], unit="s")

In [87]:
comparison_df

,Valeurs réelles,Valeurs prédites
0,47 days 00:10:38,150 days 18:30:17
1,373 days 06:06:59,316 days 04:53:24
2,457 days 03:44:50,250 days 00:14:52
3,414 days 20:01:18,410 days 01:06:20
4,270 days 16:06:29,266 days 16:16:46
...,...,...
882,194 days 15:17:58,158 days 10:14:26
883,456 days 23:40:28,451 days 08:00:04
884,398 days 22:31:20,424 days 01:30:40
885,331 days 19:00:27,313 days 16:03:26


In [89]:
comparison_df.describe()

,Valeurs réelles,Valeurs prédites
count,887,887
mean,267 days 09:15:39.377677568,267 days 11:15:13.445955468
std,152 days 11:41:55.188594082,125 days 20:42:46.004676286
min,0 days 07:28:49,-29 days +21:10:57
25%,139 days 05:07:41,172 days 00:25:50.500000
50%,270 days 16:06:29,271 days 22:48:08
75%,391 days 03:23:18.500000,366 days 11:40:54
max,567 days 15:54:47,577 days 04:19:40


In [91]:
comparison_df["Valeurs prédites"] = comparison_df["Valeurs prédites"].apply(lambda x: x if x >= pd.Timedelta(0) else pd.Timedelta(0))

In [93]:
print(comparison_df[comparison_df['Valeurs prédites'] < pd.Timedelta(0)])

Empty DataFrame
Columns: [Valeurs réelles, Valeurs prédites]
Index: []


In [95]:
comparison_df["Différence"] = abs(comparison_df["Valeurs réelles"] - comparison_df["Valeurs prédites"])
comparison_df

,Valeurs réelles,Valeurs prédites,Différence
0,47 days 00:10:38,150 days 18:30:17,103 days 18:19:39
1,373 days 06:06:59,316 days 04:53:24,57 days 01:13:35
2,457 days 03:44:50,250 days 00:14:52,207 days 03:29:58
3,414 days 20:01:18,410 days 01:06:20,4 days 18:54:58
4,270 days 16:06:29,266 days 16:16:46,3 days 23:49:43
...,...,...,...
882,194 days 15:17:58,158 days 10:14:26,36 days 05:03:32
883,456 days 23:40:28,451 days 08:00:04,5 days 15:40:24
884,398 days 22:31:20,424 days 01:30:40,25 days 02:59:20
885,331 days 19:00:27,313 days 16:03:26,18 days 02:57:01


In [97]:
comparison_df.describe()

,Valeurs réelles,Valeurs prédites,Différence
count,887,887,887
mean,267 days 09:15:39.377677568,267 days 13:51:11.080397408,48 days 18:49:36.443277903
std,152 days 11:41:55.188594082,125 days 14:56:40.664627618,49 days 15:02:19.247650019
min,0 days 07:28:49,0 days 00:00:00,0 days 04:29:44.750000
25%,139 days 05:07:41,172 days 00:25:50.500000,13 days 23:51:04.500000
50%,270 days 16:06:29,271 days 22:48:08,30 days 21:51:05
75%,391 days 03:23:18.500000,366 days 11:40:54,66 days 21:24:12
max,567 days 15:54:47,577 days 04:19:40,371 days 21:29:03


In [121]:
conf = r2_score(y_pred, y_test)
conf

0.6937045026592075

In [139]:
# Convertir les colonnes en Timedelta
comparison_df['Valeurs réelles'] = pd.to_timedelta(comparison_df['Valeurs réelles'])
comparison_df['Valeurs prédites'] = pd.to_timedelta(comparison_df['Valeurs prédites'])

# Calculer la différence en secondes
errors_in_seconds = (comparison_df['Valeurs réelles'] - comparison_df['Valeurs prédites']).dt.total_seconds()

# Calcul du R² (coefficient de détermination)
r2 = r2_score(comparison_df['Valeurs réelles'].dt.total_seconds(), comparison_df['Valeurs prédites'].dt.total_seconds())

# Calcul de la MAE (Mean Absolute Error) et RMSE (Root Mean Squared Error)
mae_seconds = mean_absolute_error(np.zeros(len(errors_in_seconds)), errors_in_seconds)
rmse_seconds = np.sqrt(mean_squared_error(np.zeros(len(errors_in_seconds)), errors_in_seconds))

# Fonction pour convertir les secondes en format jour:heure:minute:seconde
def convert_seconds_to_dhms(seconds):
    days = seconds // (24 * 3600)
    hours = (seconds % (24 * 3600)) // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    return f"{int(days)} days {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}"

# Conversion des résultats en format jour:heure:minute:seconde
mae_dhms = convert_seconds_to_dhms(mae_seconds)
rmse_dhms = convert_seconds_to_dhms(rmse_seconds)

# Affichage des résultats
print("MAE (jour:heure:minute:seconde):", mae_dhms)
print("RMSE (jour:heure:minute:seconde):", rmse_dhms)
print("R² Score:", r2)

MAE (jour:heure:minute:seconde): 48 days 18:49:36
RMSE (jour:heure:minute:seconde): 69 days 13:40:24
R² Score: 0.7916169969398325


In [141]:
test = pd.read_csv('sample_submission.csv')
test

,segment_id,time_to_eruption
0,1000213997,0
1,100023368,0
2,1000488999,0
3,1001028887,0
4,1001857862,0
...,...,...
4515,996704281,0
4516,997630809,0
4517,998072137,0
4518,998136924,0


In [143]:
folder_path2 = '/Users/alexisdafonseca/Desktop/predict-volcanic-eruptions-ingv-oe/test/'

In [145]:
# Lire un fichier au hasard pour récupérer les colonnes (assumant qu'ils ont tous les mêmes colonnes)
sample_file = f"{folder_path2}/{test['segment_id'].iloc[0]}.csv"
sample_data = pd.read_csv(sample_file)

# Initialiser le dictionnaire avec des listes vides pour chaque statistique
for col in sample_data.columns:
    for stat in columns:
        mesure[f"{col}_{stat}"] = []  

# Parcourir chaque segment_id
for file in tqdm.tqdm(test["segment_id"]):
    file_path = f"{folder_path2}/{file}.csv"

    try:
        data = pd.read_csv(file_path)

        for column in data.select_dtypes(include=[np.number]).columns:  # Vérifier si la colonne est numérique
            mesure[f"{column}_sum"].append(data[column].sum())
            mesure[f"{column}_mean"].append(data[column].mean())
            mesure[f"{column}_std"].append(data[column].std())
            mesure[f"{column}_max"].append(data[column].max())
            mesure[f"{column}_min"].append(data[column].min())

            # Ajout des quantiles demandés
            mesure[f"{column}_quantile90"].append(data[column].quantile(0.90))
            mesure[f"{column}_quantile70"].append(data[column].quantile(0.70))
            mesure[f"{column}_quantile50"].append(data[column].quantile(0.50))  # Médiane
            mesure[f"{column}_quantile20"].append(data[column].quantile(0.20))
            mesure[f"{column}_quantile10"].append(data[column].quantile(0.10))

        # Gérer les colonnes absentes en remplissant avec NaN
        missing_columns = set(sample_data.columns) - set(data.columns)
        for column in missing_columns:
            for stat in columns:
                mesure[f"{column}_{stat}"].append(np.nan)  # Remplir avec NaN au lieu de 0

    except Exception as e:
        print(f"Erreur lors de la lecture de {file_path}: {e}")
        # Ajouter NaN pour toutes les colonnes en cas d'erreur
        for key in mesure.keys():
            mesure[key].append(np.nan)

# Vérification des longueurs de listes
min_length = min(len(lst) for lst in mesure.values())
for key in mesure.keys():
    mesure[key] = mesure[key][:min_length]  # Troncature des listes trop longues

# Convertir en DataFrame
test_mesure = pd.DataFrame(mesure)
print(test_mesure.head())


100%|███████████████████████████████████████| 4520/4520 [07:43<00:00,  9.75it/s]


   sensor_1_sum  sensor_1_mean  sensor_1_std  sensor_1_min  sensor_1_max  \
0      267753.0       4.462476    306.174474       -1319.0        1347.0   
1       50287.0       0.838103    335.591100       -1268.0        1214.0   
2       47695.0       0.794903    352.172248       -3806.0        4259.0   
3      117225.0       1.953717    259.030122       -1156.0        1220.0   
4     -239079.0      -3.984584    254.306426       -1111.0         972.0   

   sensor_1_quantile90  sensor_1_quantile70  sensor_1_quantile50  \
0                393.0                155.0                  0.0   
1                435.0                181.0                  0.0   
2                392.0                151.0                  0.0   
3                333.0                138.0                  0.0   
4                317.0                131.0                  0.0   

   sensor_1_quantile20  sensor_1_quantile10  ...  sensor_10_sum  \
0               -244.0               -374.0  ...            0.0   


In [147]:
test_final = pd.concat([test, test_mesure], axis=1)  # Fusionner avec df_main
print(test_final.head())

   segment_id  time_to_eruption  sensor_1_sum  sensor_1_mean  sensor_1_std  \
0  1000213997                 0      267753.0       4.462476    306.174474   
1   100023368                 0       50287.0       0.838103    335.591100   
2  1000488999                 0       47695.0       0.794903    352.172248   
3  1001028887                 0      117225.0       1.953717    259.030122   
4  1001857862                 0     -239079.0      -3.984584    254.306426   

   sensor_1_min  sensor_1_max  sensor_1_quantile90  sensor_1_quantile70  \
0       -1319.0        1347.0                393.0                155.0   
1       -1268.0        1214.0                435.0                181.0   
2       -3806.0        4259.0                392.0                151.0   
3       -1156.0        1220.0                333.0                138.0   
4       -1111.0         972.0                317.0                131.0   

   sensor_1_quantile50  ...  sensor_10_sum  sensor_10_mean  sensor_10_std  \
0  

In [149]:
test_final = test_final.drop(columns = ['segment_id'])
test_final

,time_to_eruption,sensor_1_sum,sensor_1_mean,sensor_1_std,sensor_1_min,sensor_1_max,sensor_1_quantile90,sensor_1_quantile70,sensor_1_quantile50,sensor_1_quantile20,...,sensor_10_sum,sensor_10_mean,sensor_10_std,sensor_10_min,sensor_10_max,sensor_10_quantile90,sensor_10_quantile70,sensor_10_quantile50,sensor_10_quantile20,sensor_10_quantile10
0,0,267753.0,4.462476,306.174474,-1319.0,1347.0,393.0,155.0,0.0,-244.0,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,50287.0,0.838103,335.591100,-1268.0,1214.0,435.0,181.0,0.0,-287.0,...,-347431.0,-5.790420,569.068331,-2502.0,2820.0,718.0,291.0,0.0,-482.0,-725.0
2,0,47695.0,0.794903,352.172248,-3806.0,4259.0,392.0,151.0,0.0,-243.0,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,117225.0,1.953717,259.030122,-1156.0,1220.0,333.0,138.0,0.0,-216.0,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,-239079.0,-3.984584,254.306426,-1111.0,972.0,317.0,131.0,0.0,-215.0,...,-10698.0,-0.178297,525.529733,-2638.0,2265.0,670.0,271.0,0.0,-437.0,-669.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4515,0,-8140616.0,-135.674672,29486.243321,-32767.0,32767.0,32767.0,32767.0,0.0,-32767.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4516,0,-159989.0,-2.666439,294.261065,-1291.0,1032.0,370.0,151.0,0.0,-250.0,...,209337.0,3.488892,535.872551,-2242.0,2486.0,685.0,279.0,0.0,-441.0,-678.0
4517,0,71538.0,1.192280,371.072752,-3032.0,2641.0,449.0,175.0,0.0,-285.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4518,0,193117.0,3.218563,346.656999,-1423.0,1494.0,435.0,172.0,0.0,-278.0,...,598689.0,9.977984,808.310113,-4349.0,3898.0,1017.0,409.0,0.0,-636.0,-980.0


In [151]:
x_Test = pd.DataFrame(test_final.drop(columns = ['time_to_eruption']))

In [153]:
y_Test = random_search.best_estimator_.predict(x_Test)

In [161]:
prediction_df = pd.DataFrame({'Segment_id' : test['segment_id'], 'Time_to_eruption' : y_Test.flatten()})
prediction_df

,Segment_id,Time_to_eruption
0,1000213997,25667854.0
1,100023368,30546950.0
2,1000488999,28260340.0
3,1001028887,24020084.0
4,1001857862,14366392.0
...,...,...
4515,996704281,19709916.0
4516,997630809,15972861.0
4517,998072137,18107292.0
4518,998136924,28618152.0


In [163]:
prediction_df["Time_to_eruption"] = pd.to_timedelta(prediction_df["Time_to_eruption"], unit="s")

In [165]:
prediction_df

,Segment_id,Time_to_eruption
0,1000213997,297 days 01:57:34
1,100023368,353 days 13:15:50
2,1000488999,327 days 02:05:40
3,1001028887,278 days 00:14:44
4,1001857862,166 days 06:39:52
...,...,...
4515,996704281,228 days 02:58:36
4516,997630809,184 days 20:54:21
4517,998072137,209 days 13:48:12
4518,998136924,331 days 05:29:12


In [167]:
prediction_df.describe()

,Segment_id,Time_to_eruption
count,4.520000e+03,4520
mean,1.066993e+09,279 days 14:22:27.785970232
std,6.162904e+08,113 days 13:34:38.458234396
min,8.602880e+05,-34 days +04:25:21
25%,5.458995e+08,206 days 08:08:40
50%,1.060695e+09,293 days 05:08:17
75%,1.599284e+09,356 days 17:39:53.500000
max,2.147116e+09,601 days 01:26:40


In [169]:
prediction_df["Time_to_eruption"] = prediction_df["Time_to_eruption"].apply(lambda x: x if x >= pd.Timedelta(0) else pd.Timedelta(0))
prediction_df

,Segment_id,Time_to_eruption
0,1000213997,297 days 01:57:34
1,100023368,353 days 13:15:50
2,1000488999,327 days 02:05:40
3,1001028887,278 days 00:14:44
4,1001857862,166 days 06:39:52
...,...,...
4515,996704281,228 days 02:58:36
4516,997630809,184 days 20:54:21
4517,998072137,209 days 13:48:12
4518,998136924,331 days 05:29:12


In [171]:
print(prediction_df[prediction_df['Time_to_eruption'] < pd.Timedelta(0)])

Empty DataFrame
Columns: [Segment_id, Time_to_eruption]
Index: []


In [173]:
prediction_df

,Segment_id,Time_to_eruption
0,1000213997,297 days 01:57:34
1,100023368,353 days 13:15:50
2,1000488999,327 days 02:05:40
3,1001028887,278 days 00:14:44
4,1001857862,166 days 06:39:52
...,...,...
4515,996704281,228 days 02:58:36
4516,997630809,184 days 20:54:21
4517,998072137,209 days 13:48:12
4518,998136924,331 days 05:29:12


In [177]:
prediction_df[["Segment_id", "Time_to_eruption"]].to_csv("submission.csv", index=False)